# Hardline Tamil Narration — IndicF5 voice clone (Colab)

**Runtime → Change runtime type → GPU (T4)** before running.

Order: run **Cell 1** (installs, then auto-restarts the kernel — this is expected). After it restarts, run cells **2 → 5**. Do NOT run Cell 1 again.

Output: `hardline_audio.zip`. Unzip the wavs into `hardline-video/public/audio/clone/` locally, then run `node scripts/build-timeline.mjs`.

In [ ]:
# CELL 1 — install, then auto-restart the kernel.
# IndicF5 pins numpy<=1.26.4, which downgrades Colab's numpy; the restart makes it consistent.
!nvidia-smi -L
!pip -q install git+https://github.com/ai4bharat/IndicF5.git soundfile
!pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4"
print('Install done. Restarting kernel now — this is expected. Then run cells 2 to 5.')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

---
### ⬇️ After the kernel restarts, start from Cell 2 below. (Skip Cell 1.)

In [ ]:
# CELL 2 — sanity check numpy, then upload your reference audio (the ElevenLabs sample .mp3)
import numpy as np
print('numpy', np.__version__)  # should print 1.26.4
from google.colab import files
up = files.upload()
REF_AUDIO = list(up.keys())[0]
print('reference:', REF_AUDIO)

In [ ]:
# CELL 3 — reference transcript = EXACT words spoken in the uploaded sample.
#   (Paste the text you typed into ElevenLabs. Tamil reference gives the best Tamil clone.)
REF_TEXT = "<<< PASTE THE TRANSCRIPT OF YOUR SAMPLE HERE >>>"

# The 6 narration scenes (Tamil). Edit freely.
SCENES = {
  "s1_hook": "தமிழ்நாட்டின் அரசுப் பள்ளிகள் இன்று ஒரு பெரிய கல்வி நெருக்கடியை சந்தித்து வருகின்றன.",
  "s2_overall": "யூடைஸ் பிளஸ் அறிக்கையின்படி, ஒரே ஆண்டில் ஒரு லட்சத்து பதினைந்தாயிரம் மாணவர்கள் பொதுக் கல்வியிலிருந்து குறைந்துள்ளனர். கடந்த இரண்டு ஆண்டுகளில் மட்டும் இந்த சரிவு ஐந்து புள்ளி ஒன்பது லட்சம்.",
  "s3_govt_vs_private": "அரசுப் பள்ளிகள் ஒரே ஆண்டில் இரண்டு லட்சத்து ஏழாயிரம் மாணவர்களை இழந்துள்ளன. ஆனால் அதே நேரத்தில், தனியார் பள்ளிகளில் ஒன்றேமுக்கால் லட்சம் மாணவர்கள் புதிதாக சேர்ந்துள்ளனர்.",
  "s4_share": "இதன் விளைவாக, அரசுப் பள்ளிகளின் பங்கு முப்பத்தொன்பது சதவீதத்திலிருந்து முப்பத்தைந்து சதவீதமாக சுருங்கியுள்ளது. தனியார் பள்ளிகளோ ஐம்பது சதவீதம் என்ற பாதி அளவைத் தொட்டுவிட்டன.",
  "s5_ground": "மதுரை உசிலம்பட்டியில், அறுபத்தைந்து ஆண்டுகளாக அந்த கிராமமே சொந்த உழைப்பில் நடத்திய தொடக்கப் பள்ளி, எஞ்சியிருந்த பத்து மாணவர்களும் தனியார் பள்ளிக்கு சென்றதால், இந்த ஆண்டு நிரந்தரமாக பூட்டப்பட்டுவிட்டது.",
  "s6_outro": "பொதுக் கல்வியின் அடித்தளமே ஆட்டம் கண்டுவருகிறது. முழு செய்தியையும் படிக்க, தி ஹார்ட்லைன்."
}
print(len(SCENES), 'scenes')

In [ ]:
# CELL 4 — load model + generate every scene
import os, numpy as np, soundfile as sf, json, subprocess
from transformers import AutoModel

# IndicF5 wants a wav reference; convert the uploaded mp3.
REF_WAV = 'ref.wav'
subprocess.run(['ffmpeg','-y','-i',REF_AUDIO,'-ar','24000','-ac','1',REF_WAV],
               check=True, capture_output=True)

model = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True)
try:
    model = model.to('cuda')
except Exception as e:
    print('CPU mode:', e)

os.makedirs('out', exist_ok=True)
durations = {}
for sid, text in SCENES.items():
    audio = model(text, ref_audio_path=REF_WAV, ref_text=REF_TEXT)
    audio = np.array(audio)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    path = f'out/{sid}.wav'
    sf.write(path, audio.astype(np.float32), samplerate=24000)
    durations[sid] = round(len(audio) / 24000.0, 3)
    print(sid, durations[sid], 's')

with open('out/durations.json','w') as f:
    json.dump(durations, f, indent=2)
print('total', round(sum(durations.values()),1), 's')

In [ ]:
# CELL 5 — zip + download
import shutil
shutil.make_archive('hardline_audio', 'zip', 'out')
from google.colab import files
files.download('hardline_audio.zip')